# ch13 模型持久化与推理（练习版）

> 按提示补全 `____`，跑通所有 assert。数据、划分、端到端管道 `pipe`
> 已就绪；`pred_md5` / `cleanup` 脚手架可用。
>
> 注意：挖空块内每条语句保持**单行**；循环/分支写在挖空块之外。
> 本章会往当前目录写模型文件，结束任务会清理。

In [ ]:
from pathlib import Path

import hashlib
import os
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import joblib
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA = Path("data")

In [ ]:
# 数据与划分口径沿用 ch05：分层 8:2、median 填充、独热、标准化
dev = pd.read_csv(DATA / "defect_ml.csv")
NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"
y_all = dev[TARGET]

tr_idx, te_idx = train_test_split(np.arange(len(dev)), test_size=0.2,
                                  random_state=42, stratify=y_all)
dev_tr, dev_te = dev.iloc[tr_idx], dev.iloc[te_idx]
ys_tr, ys_te = dev_tr[TARGET], dev_te[TARGET]

imp = SimpleImputer(strategy="median").fit(dev_tr[NUM])
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(dev_tr[CAT])
Xtr = np.hstack([imp.transform(dev_tr[NUM]), oh.transform(dev_tr[CAT])])
Xte = np.hstack([imp.transform(dev_te[NUM]), oh.transform(dev_te[CAT])])
ss = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

# 端到端管道：原始 DataFrame 进 → 预测出（可整体持久化）
pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])),
    ("clf", DecisionTreeClassifier(max_depth=5, random_state=42)),
])
pipe.fit(dev_tr, ys_tr)

print("sklearn", sklearn.__version__, "| joblib", joblib.__version__,
      "| 端到端管道已拟合")

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def pred_md5(y_pred):
    """预测数组的 md5 指纹：持久化前后必须逐位一致。"""
    return hashlib.md5(np.asarray(y_pred).tobytes()).hexdigest()


def cleanup(*paths):
    """删掉本次实验落盘的模型文件。"""
    for p in paths:
        (Path(p)).unlink(missing_ok=True)


print("脚手架就绪：pred_md5 / cleanup")

## 任务 1：存档与验收

RF 存档、读档、指纹对账。

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(Xtr_s, ys_tr)
joblib.dump(rf, "rf_model.joblib")
rf2 = joblib.load("rf_model.joblib")
p_raw, p_loaded = rf.predict(Xte_s), rf2.predict(Xte_s)
md5_same = pred_md5(p_raw) == pred_md5(p_loaded)
rf_size = os.path.getsize("rf_model.joblib")
print("文件大小:", rf_size, "bytes | predict 指纹一致:", md5_same)

assert md5_same, "dump/load 往返后预测必须逐位相同"
assert rf_size == 1_359_721, "RF100 的字节级体积（同版本同数据应稳定）"
assert (p_raw == p_loaded).all()

## 任务 2：压缩存档

compress=3 的体积折扣与预测一致性。

In [ ]:
joblib.dump(rf, "rf_z.joblib", compress=3)
z_size = os.path.getsize("rf_z.joblib")
ratio = round(z_size / rf_size, 2)
joblib.dump(rf, "rf_z0.joblib")  # 对照：不压缩
z0_size = os.path.getsize("rf_z0.joblib")
print(f"压缩 {rf_size} → {z_size} bytes（{ratio}x）| 对照不压缩 {z0_size}")
print("加载后 predict 一致:", (joblib.load("rf_z.joblib").predict(Xte_s) == p_raw).all())

assert z_size == 248_622, "compress=3 的 zlib 体积"
assert ratio == 0.18, "压缩到 18%：大模型交卷前的最后一步"
assert (joblib.load("rf_z.joblib").predict(Xte_s) == p_raw).all(), "压缩不改变预测"

## 任务 3：读档完整性体检

属性、超参、概率逐项对账。

In [ ]:
n_feat = rf2.n_features_in_
n_est = rf2.n_estimators
gp_diff = {k for k, v in rf.get_params(deep=False).items() if not np.array_equal(np.asarray(v, dtype=object), np.asarray(rf2.get_params(deep=False)[k], dtype=object))}
proba_same = bool(np.allclose(rf.predict_proba(Xte_s), rf2.predict_proba(Xte_s)))
print("n_features_in_:", n_feat, "| n_estimators:", n_est)
print("get_params 差异项:", gp_diff or "无", "| proba 一致:", proba_same)

assert n_feat == 16 and n_est == 100
assert gp_diff == set(), "加载版与原版超参完全一致"
assert proba_same, "predict_proba 也逐位一致"

## 任务 4：端到端管道持久化

原始 DataFrame 进，预测出。

In [ ]:
pipe_acc = round(float(accuracy_score(ys_te, pipe.predict(dev_te))), 4)
joblib.dump(pipe, "pipe.joblib")
pipe2 = joblib.load("pipe.joblib")
p_pipe = pipe.predict(dev_te)
p_pipe2 = pipe2.predict(dev_te)
pipe_size = os.path.getsize("pipe.joblib")
print("端到端 acc:", pipe_acc, "| 文件:", pipe_size, "bytes | 重载一致:",
      bool((p_pipe == p_pipe2).all()))

assert pipe_acc == 0.7917, "端到端管道成绩与 ch05 的 d5 树一致"
assert pipe_size == 7_556, "整条管道只有 7.6KB：预处理状态+树全在里面"
assert (p_pipe == p_pipe2).all(), "管道往返同样逐位一致"
assert list(pipe2.predict(dev_te.iloc[:3])) == pipe.predict(dev_te.iloc[:3]).tolist()

## 任务 5：单样本在线推理

新台区一进来就给结论，未知类别兜底。

In [ ]:
new = pd.DataFrame([{"设备类型": "变压器", "缺陷类型": "发热", "负荷值": 500.0,
                     "温度": 45.0, "湿度": 70.0, "投运年限": 20, "巡检耗时分钟": 60}])
proba_new = np.round(pipe2.predict_proba(new)[0], 4).tolist()
pred_new = int(pipe2.predict(new)[0])
weird = new.copy()
weird["设备类型"] = "巡检机器人"  # 训练时没见过的类别
proba_weird = np.round(pipe2.predict_proba(weird)[0], 4).tolist()
print("新样本 proba:", proba_new, "→ 判为", pred_new)
print("未知设备类型 proba:", proba_weird, "→ handle_unknown=ignore 兜底不炸")

assert pred_new == 1, "变压器+发热+高负荷 → 危急"
assert proba_new == [0.1111, 0.8889]
assert proba_weird == [0.1111, 0.8889], "未知类别被独热归零后走同一条树叶路径"

## 任务 6：版本戳

在 pickle 字节里找到 `_sklearn_version`。

In [ ]:
raw_bytes = Path("rf_model.joblib").read_bytes()
has_version = b"_sklearn_version" in raw_bytes
pos = raw_bytes.find(b"_sklearn_version")
version_bytes = raw_bytes[pos:pos + 40]
print("含 _sklearn_version 戳:", has_version)
print("戳附近字节:", version_bytes)
print("当前 sklearn:", sklearn.__version__, "——跨版本 load 前先核对戳")

assert has_version, "sklearn 1.9 起每个持久化对象都带版本戳"
assert version_bytes.startswith(b"_sklearn_version")

## 任务 7：推理脚本模拟

load → 批量预测 → 指纹对账的最小闭环。

In [ ]:
prod_model = joblib.load("rf_model.joblib")
batch = pd.read_csv(DATA / "defect_ml.csv").iloc[:50]  # 模拟新来的数据（带缺失/原始列）
prod_pred = prod_model.predict(
    np.hstack([imp.transform(batch[NUM]), oh.transform(batch[CAT])]))
prod_md5 = pred_md5(prod_pred)
raw_md5 = pred_md5(rf.predict(
    np.hstack([imp.transform(batch[NUM]), oh.transform(batch[CAT])])))
print("批大小:", len(prod_pred), "| 指纹:", prod_md5[:16], "| 与训练时模型一致:", prod_md5 == raw_md5)

assert len(prod_pred) == 50
assert prod_md5 == raw_md5, "推理侧产物与训练侧逐位相同——部署闭环成立"

## 任务 8：原生 pickle 对比

体积、结果、版本戳三项对照 joblib。

In [ ]:
import pickle
rf3 = joblib.load("rf_model.joblib")
pk_fh = open("rf_pickle.pkl", "wb")
pickle.dump(rf3, pk_fh)
pk_fh.close()
pk_size = os.path.getsize("rf_pickle.pkl")
rf4 = pickle.load(open("rf_pickle.pkl", "rb"))
pk_same = bool((rf4.predict(Xte_s) == p_raw).all())
pk_has_version = b"_sklearn_version" in Path("rf_pickle.pkl").read_bytes()
print(f"原生 pickle {pk_size} bytes（joblib {rf_size}）| 预测一致: {pk_same} | 含版本戳: {pk_has_version}")
print("结论: 两者体积接近、结果相同，但 joblib 对 numpy 大数组更快且支持压缩，"
      "又是 sklearn 生态约定俗成——竞赛交模型一律 joblib")

assert pk_same
assert pk_has_version
assert pk_size < rf_size, "原生 pickle 略小于 joblib（差 ~8KB 对齐开销），但同一量级"

## 完成标准

所有 assert 通过 = 任务完成。最后清理落盘文件：

In [ ]:
# 收尾清理（不挖空）：删掉本章实验文件
cleanup("rf_model.joblib", "rf_z.joblib", "rf_z0.joblib",
        "pipe.joblib", "rf_pickle.pkl")
import glob
print("残留模型文件:", glob.glob("*.joblib") + glob.glob("*.pkl"))

## 完成标准（续）

自查：为什么验收要指纹、为什么只存树不存预处理是事故、
版本戳怎么用、推理脚本的三条纪律是什么。